# Build derived TorNet modeling manifests

Build a validated modeling manifest from the preserved raw
annual audit without rescanning or modifying the source archive.

The official test set is preserved exactly. Explicitly documented
training exclusions resolve confirmed train/test event leakage.


In [15]:
%pip install -q xarray netCDF4 pandas pyarrow


In [16]:
from google.colab import drive

drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
%pip install --force-reinstall --no-deps "/content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.3-py3-none-any.whl"


Processing ./drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.3-py3-none-any.whl
  Attempting uninstall: tornet-detection
    Found existing installation: tornet-detection 0.1.3
    Uninstalling tornet-detection-0.1.3:
      Successfully uninstalled tornet-detection-0.1.3


In [18]:
from pathlib import Path
import json

import pandas as pd

import tornado_detection

from tornado_detection.data.modeling import (
    ModelingExclusion,
    build_modeling_manifest,
    write_modeling_manifest_artifacts,
)

assert tornado_detection.__version__ == "0.1.3"

print(
    "tornado_detection package version:",
    tornado_detection.__version__,
)
print(
    "Loaded tornado_detection from:",
    tornado_detection.__file__,
)


tornado_detection package version: 0.1.3
Loaded tornado_detection from: /usr/local/lib/python3.12/dist-packages/tornado_detection/__init__.py


## 2021 leakage resolution

Preserve every official test file and exclude only the 17 training
files participating in confirmed events `956483` and `962857`.


In [19]:
RAW_DIRECTORY = Path(
    "/content/drive/MyDrive/TorNet_Backup/manifests/v1/2021"
)

OUTPUT_DIRECTORY = Path(
    "/content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2021"
)

EXPECTED_DIMENSIONS = {
    "time": 4,
    "sweep": 2,
    "azimuth": 120,
    "range": 240,
    "lims": 2,
}

EXCLUSION_SPECS = (
    (
        (
            "train/2021/"
            "WRN_210516_233723_KPUX_1085609n_F6.nc"
        ),
        "956483",
        "158215",
    ),
    (
        (
            "train/2021/"
            "WRN_210516_234424_KPUX_1085609n_F6.nc"
        ),
        "956483",
        "158215",
    ),
    (
        (
            "train/2021/"
            "WRN_210516_233022_KPUX_1085609n_K5.nc"
        ),
        "962857",
        "159188",
    ),
    (
        (
            "train/2021/"
            "WRN_210516_233022_KPUX_1085610n_K5.nc"
        ),
        "962857",
        "159188",
    ),
    (
        (
            "train/2021/"
            "WRN_210516_233723_KPUX_1085610n_K5.nc"
        ),
        "962857",
        "159188",
    ),
    (
        (
            "train/2021/"
            "WRN_210516_234424_KPUX_1085610n_K5.nc"
        ),
        "962857",
        "159188",
    ),
    (
        (
            "train/2021/"
            "WRN_210516_235112_KPUX_1085609n_F6.nc"
        ),
        "962857",
        "159188",
    ),
    (
        (
            "train/2021/"
            "WRN_210516_235112_KPUX_1085610n_K5.nc"
        ),
        "962857",
        "159188",
    ),
    (
        (
            "train/2021/"
            "WRN_210516_235749_KPUX_1085609n_F6.nc"
        ),
        "962857",
        "159188",
    ),
    (
        (
            "train/2021/"
            "WRN_210516_235749_KPUX_1085610n_K5.nc"
        ),
        "962857",
        "159188",
    ),
    (
        (
            "train/2021/"
            "WRN_210517_000427_KPUX_1085610n_K5.nc"
        ),
        "962857",
        "159188",
    ),
    (
        (
            "train/2021/"
            "WRN_210517_001103_KPUX_1085610n_K5.nc"
        ),
        "962857",
        "159188",
    ),
    (
        (
            "train/2021/"
            "WRN_210517_001741_KPUX_1085610n_K5.nc"
        ),
        "962857",
        "159188",
    ),
    (
        (
            "train/2021/"
            "WRN_210517_002419_KPUX_1085610n_K5.nc"
        ),
        "962857",
        "159188",
    ),
    (
        (
            "train/2021/"
            "WRN_210517_003121_KPUX_1085610n_K5.nc"
        ),
        "962857",
        "159188",
    ),
    (
        (
            "train/2021/"
            "WRN_210517_003739_KPUX_1085610n_K5.nc"
        ),
        "962857",
        "159188",
    ),
    (
        (
            "train/2021/"
            "WRN_210517_004358_KPUX_1085610n_K5.nc"
        ),
        "962857",
        "159188",
    ),
)

EXCLUSIONS = [
    ModelingExclusion(
        archive_member=archive_member,
        expected_event_id=event_id,
        expected_episode_id=episode_id,
        reason=(
            "Official train/test event leakage"
        ),
        resolution=(
            "Preserve every official test file and "
            "exclude the overlapping training members"
        ),
    )
    for (
        archive_member,
        event_id,
        episode_id,
    ) in EXCLUSION_SPECS
]

assert len(EXCLUSION_SPECS) == 17
assert len(
    {
        archive_member
        for archive_member, _, _ in EXCLUSION_SPECS
    }
) == 17

print("Raw manifest:", RAW_DIRECTORY)
print("Modeling output:", OUTPUT_DIRECTORY)
print("Configured exclusions:", len(EXCLUSIONS))


Raw manifest: /content/drive/MyDrive/TorNet_Backup/manifests/v1/2021
Modeling output: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2021
Configured exclusions: 17


In [20]:
modeling_build = build_modeling_manifest(
    RAW_DIRECTORY,
    exclusions=EXCLUSIONS,
    expected_dimensions=EXPECTED_DIMENSIONS,
)

ledger = modeling_build.exclusion_ledger
validation = modeling_build.validation
manifest = modeling_build.manifest

display(ledger)
display(validation.checks)

print(
    "Event groups crossing official splits:"
)
display(validation.event_split_overlap)

print(
    "Episode groups crossing official splits "
    "(informational):"
)
display(validation.episode_split_overlap)

expected_members = [
    archive_member
    for archive_member, _, _ in EXCLUSION_SPECS
]

assert manifest.year == 2021
assert (
    modeling_build.source_terminal_marker
    == "_INVALID.json"
)

assert len(ledger) == 17
assert (
    ledger["archive_member"].tolist()
    == expected_members
)

assert set(ledger["split"]) == {"train"}
assert set(ledger["category"]) == {"WRN"}
assert set(ledger["radar_site"]) == {"KPUX"}

actual_event_counts = (
    ledger["event_id"]
    .astype(str)
    .value_counts()
    .sort_index()
    .to_dict()
)

assert actual_event_counts == {
    "956483": 2,
    "962857": 15,
}

actual_event_episode_pairs = set(
    zip(
        ledger["event_id"].astype(str),
        ledger["episode_id"].astype(str),
        strict=True,
    )
)

assert actual_event_episode_pairs == {
    ("956483", "158215"),
    ("962857", "159188"),
}

assert set(
    ledger["removed_frame_count"].astype(int)
) == {4}

assert int(
    ledger["removed_frame_count"].sum()
) == 68

assert len(
    manifest.file_manifest
) == 23_300

assert len(
    manifest.frame_manifest
) == 93_200

assert (
    manifest.netcdf_member_count
    == 23_300
)

assert validation.all_required_passed
assert validation.event_split_overlap.empty

excluded_positive_frames = int(
    ledger[
        "removed_positive_frame_count"
    ].sum()
)

print(
    "Excluded positive frames:",
    excluded_positive_frames,
)

print(
    "PASS: 2021 modeling manifest validated "
    "before writing"
)


,manifest_schema_version,year,archive_member,file_id,split,category,event_id,episode_id,radar_site,removed_frame_count,removed_positive_frame_count,reason,resolution
0,1.0.0,2021,train/2021/WRN_210516_233723_KPUX_1085609n_F6.nc,3a374df06218d50dc74fbbfbb1c9ef17e1e8cea45aede4...,train,WRN,956483,158215,KPUX,4,0,Official train/test event leakage,Preserve every official test file and exclude ...
1,1.0.0,2021,train/2021/WRN_210516_234424_KPUX_1085609n_F6.nc,414fec1d259f3eef9d396a633835fbfdb72bca7a142e80...,train,WRN,956483,158215,KPUX,4,0,Official train/test event leakage,Preserve every official test file and exclude ...
2,1.0.0,2021,train/2021/WRN_210516_233022_KPUX_1085609n_K5.nc,ca5e449e98eb45d2a28b5e21fad3138393577034a2892f...,train,WRN,962857,159188,KPUX,4,0,Official train/test event leakage,Preserve every official test file and exclude ...
3,1.0.0,2021,train/2021/WRN_210516_233022_KPUX_1085610n_K5.nc,a70f9f0c00c8e2d703d50c5532fceb2b372064575ed6ca...,train,WRN,962857,159188,KPUX,4,0,Official train/test event leakage,Preserve every official test file and exclude ...
4,1.0.0,2021,train/2021/WRN_210516_233723_KPUX_1085610n_K5.nc,0bf25a73d7fbc0d219b48c6e20e6d9b830617df1736a59...,train,WRN,962857,159188,KPUX,4,0,Official train/test event leakage,Preserve every official test file and exclude ...
5,1.0.0,2021,train/2021/WRN_210516_234424_KPUX_1085610n_K5.nc,b5ac8387c15527b4f5dec13fef5f14a7e543c2e040a4bb...,train,WRN,962857,159188,KPUX,4,0,Official train/test event leakage,Preserve every official test file and exclude ...
6,1.0.0,2021,train/2021/WRN_210516_235112_KPUX_1085609n_F6.nc,0bb37c7613e043218ea2244a5dd1f2b468ae667226916d...,train,WRN,962857,159188,KPUX,4,0,Official train/test event leakage,Preserve every official test file and exclude ...
7,1.0.0,2021,train/2021/WRN_210516_235112_KPUX_1085610n_K5.nc,21650c65f3f377808930cd29129fa1e29b71fdb5132631...,train,WRN,962857,159188,KPUX,4,0,Official train/test event leakage,Preserve every official test file and exclude ...
8,1.0.0,2021,train/2021/WRN_210516_235749_KPUX_1085609n_F6.nc,19581c37995e806b2b74581191f0034b589c2338250b20...,train,WRN,962857,159188,KPUX,4,0,Official train/test event leakage,Preserve every official test file and exclude ...
9,1.0.0,2021,train/2021/WRN_210516_235749_KPUX_1085610n_K5.nc,e3e5c887f15c391e99361ec8e47370e3a6197d0e8f5aae...,train,WRN,962857,159188,KPUX,4,0,Official train/test event leakage,Preserve every official test file and exclude ...


,check,required,passed,observed,expected,detail
0,build_errors,True,True,0,0,
1,file_row_count,True,True,23300,23300,
2,frame_row_count,True,True,93200,93200,
3,unique_archive_members,True,True,23300,23300,
4,unique_file_ids,True,True,23300,23300,
5,unique_frame_ids,True,True,93200,93200,
6,frame_rows_match_file_frame_counts,True,True,0,0,
7,frame_label_sums_match_file_manifest,True,True,0,0,
8,frame_indices_are_contiguous,True,True,0,0,
9,expected_frames_per_file,True,True,0,0,Expected 4 frames for every file


Event groups crossing official splits:


,event_group_id,splits_json,file_count,archive_members_json


Episode groups crossing official splits (informational):


,episode_id,splits_json,file_count,archive_members_json
0,156736,"[""test"",""train""]",11,"[""test/2021/TOR_210318_090718_KEOX_947198_Y3.n..."
1,157634,"[""test"",""train""]",14,"[""test/2021/NUL_210409_161816_KSHV_953124s_F2...."
2,158880,"[""test"",""train""]",10,"[""test/2021/NUL_210517_044252_KFWS_961217s_W9...."
3,159123,"[""test"",""train""]",56,"[""test/2021/NUL_210607_084844_KLZK_962306s_B2...."
4,159188,"[""test"",""train""]",31,"[""test/2021/WRN_210517_003253_KGLD_1085612n_V8..."
5,161323,"[""test"",""train""]",44,"[""test/2021/WRN_210916_221704_KMPX_1086485n_F0..."
6,163059,"[""test"",""train""]",31,"[""test/2021/WRN_210916_232249_KMPX_1086485n_J1..."
7,163230,"[""test"",""train""]",23,"[""test/2021/WRN_211006_111335_KEVX_1086505n_M9..."
8,165321,"[""test"",""train""]",12,"[""test/2021/WRN_211205_234621_KNQA_1086686n_B5..."
9,189997,"[""test"",""train""]",46,"[""test/2021/NUL_210517_002744_KGLD_1085612n_T7..."


Excluded positive frames: 0
PASS: 2021 modeling manifest validated before writing


In [21]:
artifacts = write_modeling_manifest_artifacts(
    modeling_build,
    OUTPUT_DIRECTORY,
    overwrite=False,
)

for artifact_name, artifact_path in sorted(
    artifacts.items()
):
    print(f"- {artifact_name}: {artifact_path}")

required_artifacts = {
    "file_manifest.parquet",
    "frame_manifest.parquet",
    "schema_summary.csv",
    "build_errors.csv",
    "validation_checks.csv",
    "event_split_overlap.csv",
    "episode_split_overlap.csv",
    "category_frame_summary.csv",
    "manifest_summary.json",
    "exclusion_ledger.csv",
    "modeling_summary.json",
    "_SUCCESS.json",
}

assert required_artifacts.issubset(
    artifacts.keys()
)

assert "_INVALID.json" not in artifacts

summary = json.loads(
    (
        OUTPUT_DIRECTORY
        / "modeling_summary.json"
    ).read_text()
)

assert summary["year"] == 2021
assert summary["status"] == "valid"

assert (
    summary["source_raw_terminal_marker"]
    == "_INVALID.json"
)

assert summary["source_file_row_count"] == 23_317
assert summary["source_frame_row_count"] == 93_268

assert summary["modeling_file_row_count"] == 23_300
assert summary["modeling_frame_row_count"] == 93_200

assert summary["excluded_file_count"] == 17
assert summary["excluded_frame_count"] == 68

assert (
    summary["excluded_positive_frame_count"]
    == int(
        ledger[
            "removed_positive_frame_count"
        ].sum()
    )
)

assert summary["event_split_overlap_count"] == 0

assert (
    summary["all_required_validations_passed"]
    is True
)

print(
    "Excluded positive frames written:",
    summary["excluded_positive_frame_count"],
)

print(
    "PASS: 2021 modeling manifest built "
    "and written"
)


- _SUCCESS.json: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2021/_SUCCESS.json
- build_errors.csv: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2021/build_errors.csv
- category_frame_summary.csv: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2021/category_frame_summary.csv
- episode_split_overlap.csv: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2021/episode_split_overlap.csv
- event_split_overlap.csv: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2021/event_split_overlap.csv
- exclusion_ledger.csv: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2021/exclusion_ledger.csv
- file_manifest.parquet: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2021/file_manifest.parquet
- frame_manifest.parquet: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2021/frame_manifest.parquet
- manifest_summary.json: /content/drive/MyDrive/TorNet_Backup/manifests/modeling/v1/2021/manifest_summary.json
- mod